# PCGT Colab Runner (Drive Data, No Internet Download)

This notebook is cleaned for a simple workflow:

1. Clone or update the repo.
2. Install Python dependencies only (no dataset downloads).
3. Mount Google Drive once.
4. Use dataset directly from Drive via symlink.
5. Verify GPU.
6. Run either a quick test or all 7 datasets.

Assumption: your dataset is already present at `/content/drive/MyDrive/PCGT/data` with the same structure as local `data/`.

## 1) Clone Or Update Repository

Run this once per session to ensure code is current.

In [1]:
import os

if not os.path.exists('/content/PCGT'):
    !git clone https://github.com/ranjanchoubey/PCGT.git /content/PCGT
else:
    %cd /content/PCGT
    !git pull
    %cd /content

%cd /content/PCGT

Cloning into '/content/PCGT'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (257/257), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 257 (delta 113), reused 242 (delta 100), pack-reused 0 (from 0)
Receiving objects: 100% (257/257), 9.11 MiB | 35.88 MiB/s, done.
Resolving deltas: 100% (113/113), done.
/content/PCGT


## 2) Install Dependencies

This installs code dependencies only. It does not download datasets.

In [2]:
import torch

# Uninstall previous versions to prevent conflicts
!pip uninstall torch-scatter torch-sparse torch-geometric torch-cluster --y

# Install compatible versions using the PyG index URL
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install get-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html

# Optionally, install PyTorch Geometric itself
!pip install git+https://github.com/pyg-team/pytorch_geometric.git


Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 106.4 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 54.4 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
ERROR: Could not find a version that satisfies the requirement get-cluster (from versions: none)
ERROR: No matching distribution found for get-cluster
  Cloning https://github.com/pyg-team/pytorch_geometric.git to /tmp/pip-req-build-ns809uoq
  Running command git clone --filter=blob:none --quiet https://github.com/pyg-team/pytorch_geometric.git /tmp/pip-req-build-ns809uoq
  Resolved https://github.com/pyg-team/pytorch_geometric.git to commit 9df668c92ecc561190a93eb7f59f6a9d47640535
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel f

In [3]:
! pip install torch-sparse

In [4]:
! pip install performer_pytorch

## 3) Mount Drive And Use Data In-Place

This mounts Drive and creates `/content/PCGT/data` as a symlink to your Drive dataset.

No data copy is performed.

In [7]:
import os
import re
from datetime import datetime
from pathlib import Path
from google.colab import drive

# 1) Mount drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive', force_remount=False)
else:
    print('Drive already mounted.')

# ✅ IMPORTANT: Direct override (your actual dataset location)
DATASET_ROOT_OVERRIDE = '/content/drive/MyDrive/PCGT_datasets/data'

MYDRIVE = Path('/content/drive/MyDrive')
LOCAL_DATA_ROOT = Path('/content/PCGT/data')
FORCE_DRIVE_LINK = True

# ✅ Accept ANY dataset (not only Planetoid etc.)
POSSIBLE_DATASETS = ['deezer', 'Planetoid', 'geom-gcn', 'wiki_new']


def has_expected_structure(p: Path) -> bool:
    if not p.exists() or not p.is_dir():
        return False
    return any((p / k).exists() for k in POSSIBLE_DATASETS)


def resolve_data_root(base: Path):
    if has_expected_structure(base / 'data'):
        return base / 'data'
    if has_expected_structure(base):
        return base
    return None


# -------------------------------
# Resolve dataset root
# -------------------------------
DRIVE_DATA_ROOT = None
checked = []

# A) Override path (MAIN FIX)
if DATASET_ROOT_OVERRIDE:
    p = Path(DATASET_ROOT_OVERRIDE).expanduser()
    checked.append(str(p))
    DRIVE_DATA_ROOT = resolve_data_root(p)

# B) Fallback search (optional)
if DRIVE_DATA_ROOT is None:
    print('Fallback search running...')
    for root, dirs, _ in os.walk('/content/drive'):
        root_p = Path(root)
        r = resolve_data_root(root_p)
        if r is not None:
            DRIVE_DATA_ROOT = r
            print('Found dataset at:', r)
            break

# C) Fail if not found
if DRIVE_DATA_ROOT is None:
    print('❌ Dataset not found.')
    print('Checked paths:')
    for p in checked:
        print(' -', p)
    raise RuntimeError('Dataset not found.')

# -------------------------------
# Create symlink (CRITICAL PART)
# -------------------------------
LOCAL_DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)

if LOCAL_DATA_ROOT.exists() or LOCAL_DATA_ROOT.is_symlink():
    if FORCE_DRIVE_LINK:
        !rm -rf /content/PCGT/data

LOCAL_DATA_ROOT.symlink_to(DRIVE_DATA_ROOT)

# -------------------------------
# Debug prints
# -------------------------------
print('\n✅ Drive data root  :', DRIVE_DATA_ROOT)
print('✅ Local data path  :', LOCAL_DATA_ROOT, '->', LOCAL_DATA_ROOT.resolve())

print('\n📂 Available datasets:')
!ls /content/PCGT/data

print('\n📂 Deezer contents:')
!ls /content/PCGT/data/deezer

Drive already mounted.

✅ Drive data root  : /content/drive/MyDrive/PCGT_datasets/data
✅ Local data path  : /content/PCGT/data -> /content/drive/.shortcut-targets-by-id/1OTepte3wFTL5aGFcn366SwoMi0Hm_cQF/PCGT_datasets/data

📂 Available datasets:
deezer

📂 Deezer contents:
deezer-europe.mat


## 5) Verify GPU

Run this before training.

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda :', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi | head -n 15

## 6) Optional Full Run (All 7 Datasets)

Runs all final configs and saves logs to Drive.

In [ ]:
%cd /content/PCGT/medium

import os
from datetime import datetime

DATA_DIR = '/content/PCGT/data/'  # keep trailing slash
RESULTS_DIR = '/content/drive/MyDrive/PCGT/results_colab'
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = f"{RESULTS_DIR}/{RUN_TAG}"
os.makedirs(RUN_DIR, exist_ok=True)

print('Data dir   :', DATA_DIR)
print('Results dir:', RUN_DIR)

# Cora

In [ ]:
!python -B main.py --method pcgt --dataset cora --lr 0.01 --num_layers 4 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --rand_split_class --label_num_per_class 20 --valid_num 500 --test_num 1000 --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 7 --graph_weight 0.8 --dropout 0.4 --weight_decay 5e-4 --ours_weight_decay 0.001 --ours_dropout 0.2 2>&1 | tee {RUN_DIR}/cora_pcgt_gpu.txt

# CiteSeer


In [ ]:
!python -B main.py --method pcgt --dataset citeseer --lr 0.01 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --rand_split_class --label_num_per_class 20 --valid_num 500 --test_num 1000 --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 7 --graph_weight 0.7 --dropout 0.5 --weight_decay 0.01 --ours_weight_decay 0.02 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/citeseer_pcgt_gpu.txt

# PubMed


In [ ]:
!python -B main.py --method pcgt --dataset pubmed --lr 0.01 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --rand_split_class --label_num_per_class 20 --valid_num 500 --test_num 1000 --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 50 --graph_weight 0.8 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/pubmed_pcgt_gpu.txt

# Chameleon

In [ ]:
!python -B main.py --method pcgt --dataset chameleon --lr 0.01 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 5 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 10 --graph_weight 0.8 --dropout 0.5 --weight_decay 0.001 --ours_weight_decay 0.01 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/chameleon_pcgt_gpu.txt

# Film

In [ ]:
!python -B main.py --method pcgt --dataset film --lr 0.05 --num_layers 2 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --seed 123 --device 0 --epochs 500 --patience 200 --runs 10 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 5 --graph_weight 0.5 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/film_pcgt_gpu.txt

# Squirrel

In [ ]:
!python -B main.py --method pcgt --dataset squirrel --lr 0.01 --num_layers 4 --hidden_channels 64 --ours_layers 1 --num_reps 4 --partition_method metis --use_graph --use_residual --backbone gcn --no_feat_norm --seed 123 --device 0 --epochs 500 --patience 200 --runs 10 --display_step 100 --aggregate add --data_dir {DATA_DIR} --num_partitions 10 --graph_weight 0.8 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 2>&1 | tee {RUN_DIR}/squirrel_pcgt_gpu.txt

# Deezer

In [ ]:
%cd /content/PCGT/medium
import os

DZ_DIR = f"{RUN_DIR}/deezer_attack"
os.makedirs(DZ_DIR, exist_ok=True)

# Common args for all Deezer runs
BASE = (
    "python -B main.py --method pcgt --dataset deezer-europe "
    "--use_graph --use_residual --backbone gcn --rand_split --seed 123 "
    f"--device 0 --epochs 500 --patience 200 --runs 3 --display_step 100 --aggregate add --data_dir {DATA_DIR}"
)

# ──────────────────────────────────────────────────────────────
# DZ-B1: Capacity reduction — hidden=32, reps=2
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B1: hidden=32, reps=2 ===")
!{BASE} --hidden_channels 32 --num_layers 2 --ours_layers 1 --num_reps 2 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.6 --weight_decay 5e-5 --ours_weight_decay 0.01 --ours_dropout 0.4 \
  2>&1 | tee {DZ_DIR}/DZ-B1_h32_reps2.txt

# ──────────────────────────────────────────────────────────────
# DZ-B2: Even smaller — hidden=32, reps=1, ours_layers=1
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B2: hidden=32, reps=1, minimal ===")
!{BASE} --hidden_channels 32 --num_layers 2 --ours_layers 1 --num_reps 1 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.6 --weight_decay 1e-4 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B2_h32_reps1.txt

# ──────────────────────────────────────────────────────────────
# DZ-B3: GCN layers=1 (simpler backbone)
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B3: num_layers=1, h=64 ===")
!{BASE} --hidden_channels 64 --num_layers 1 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.6 --weight_decay 5e-5 --ours_weight_decay 0.01 --ours_dropout 0.4 \
  2>&1 | tee {DZ_DIR}/DZ-B3_L1.txt

In [ ]:
# ──────────────────────────────────────────────────────────────
# DZ-B4: Heavy dropout=0.7, stronger wd
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B4: dropout=0.7, wd=5e-3 ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B4_d07_wd5e3.txt

# ──────────────────────────────────────────────────────────────
# DZ-B5: Very heavy dropout=0.8 + wd=0.01
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B5: dropout=0.8, wd=0.01 ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.8 --weight_decay 0.01 --ours_weight_decay 0.05 --ours_dropout 0.6 \
  2>&1 | tee {DZ_DIR}/DZ-B5_d08_wd01.txt

# ──────────────────────────────────────────────────────────────
# DZ-B6: Moderate dropout=0.7 + small hidden=32
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B6: h=32, d=0.7, wd=5e-3 ===")
!{BASE} --hidden_channels 32 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B6_h32_d07.txt

In [ ]:
# ──────────────────────────────────────────────────────────────
# DZ-B7: BatchNorm + gw=0.9 (heavy GCN)
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B7: use_bn, gw=0.9 ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.9 --use_bn \
  --lr 0.01 --dropout 0.6 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.4 \
  2>&1 | tee {DZ_DIR}/DZ-B7_bn_gw09.txt

# ──────────────────────────────────────────────────────────────
# DZ-B8: BatchNorm + gw=0.3 (heavy PCGT)
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B8: use_bn, gw=0.3 ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.3 --use_bn \
  --lr 0.01 --dropout 0.6 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.4 \
  2>&1 | tee {DZ_DIR}/DZ-B8_bn_gw03.txt

# ──────────────────────────────────────────────────────────────
# DZ-B9: KMeans K=20 (larger partitions, more nodes per cluster)
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B9: kmeans K=20 ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 20 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B9_K20.txt

# ──────────────────────────────────────────────────────────────
# DZ-B10: Random partitioning (ablation — are KMeans partitions actually helping?)
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B10: random partition K=50 ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method random --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B10_random.txt

In [ ]:
# ──────────────────────────────────────────────────────────────
# DZ-B11: No feature normalization
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B11: no_feat_norm ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 0.5 --no_feat_norm \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B11_nofeatnorm.txt

# ──────────────────────────────────────────────────────────────
# DZ-B12: Pure GCN baseline (no PCGT attention at all)
# graph_weight=1.0 means output = 1.0*GCN + 0.0*PCGT
# This tells us the GCN ceiling on Deezer
# ──────────────────────────────────────────────────────────────
print("\n=== DZ-B12: pure GCN (gw=1.0) ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method kmeans --num_partitions 50 --graph_weight 1.0 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {DZ_DIR}/DZ-B12_pure_gcn.txt

In [ ]:
# ──────────────────────────────────────────────────────────────
# SUMMARY: Parse all Deezer attack results
# ──────────────────────────────────────────────────────────────
import re, glob

print("="*70)
print("  DEEZER ATTACK RESULTS SUMMARY")
print("="*70)
print(f"{'Config':<30} {'Highest Test':<20} {'Final Test':<20}")
print("-"*70)

results = []
for f in sorted(glob.glob(f"{DZ_DIR}/DZ-B*.txt")):
    name = os.path.basename(f).replace('.txt','')
    with open(f) as fh:
        text = fh.read()
    m = re.search(r'Highest Test:\s+([\d.]+)\s*±\s*([\d.]+)', text)
    m2 = re.search(r'Final Test:\s+([\d.]+)\s*±\s*([\d.]+)', text)
    if m:
        ht = f"{m.group(1)} ± {m.group(2)}"
        ft = f"{m2.group(1)} ± {m2.group(2)}" if m2 else "N/A"
        results.append((name, float(m.group(1)), ht, ft))
        print(f"{name:<30} {ht:<20} {ft:<20}")
    else:
        print(f"{name:<30} {'PARSING ERROR':<20}")

if results:
    best = max(results, key=lambda x: x[1])
    print("-"*70)
    print(f"BEST: {best[0]} → {best[2]}")
    print(f"\nPrevious best: 64.94 ± 0.85  |  SGFormer target: 67.1 ± 1.1")

## After the sweep: Run the best config with 5 runs

Copy the best config from above and change `--runs 3` to `--runs 5` below.

In [ ]:
# ──────────────────────────────────────────────────────────────
# BEST CONFIG: DZ-B10 (random partition K=50) — 5 runs
# 3-run probe: 67.18 ± 0.76  (matches SGFormer 67.1 ± 1.1)
# ──────────────────────────────────────────────────────────────

print("\n=== DZ-B10 BEST — 5 runs ===")
!{BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method random --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  --runs 5 \
  2>&1 | tee {DZ_DIR}/DZ-B10_BEST_5run.txt

---
# Phase 2: Cora + Actor Attack Sweep (3-run probes)

**Goal**: Beat SGFormer with wider margins on **Cora** (83.80 → 84.5+) and **Actor** (37.69 → 37.9+)

**Untested levers being probed:**
- `--num_heads 4` (multi-head attention)
- `--ours_layers 2` (deeper PCGT)
- `--aggregate cat` (concatenation fusion)
- `--partition_method spectral/random` (alternative partitioning)
- `--backbone gat` (GAT instead of GCN)
- Different `--num_partitions`

In [ ]:
%cd /content/PCGT/medium
import os

ATTACK_DIR = f"{RUN_DIR}/phase2_attack"
os.makedirs(f"{ATTACK_DIR}/cora", exist_ok=True)
os.makedirs(f"{ATTACK_DIR}/actor", exist_ok=True)
os.makedirs(f"{ATTACK_DIR}/pubmed", exist_ok=True)
os.makedirs(f"{ATTACK_DIR}/deezer", exist_ok=True)

# Base commands per dataset (3-run probes on GPU)
CORA_BASE = (
    "python -B main.py --method pcgt --dataset cora "
    "--use_graph --use_residual --backbone gcn --rand_split_class "
    "--label_num_per_class 20 --valid_num 500 --test_num 1000 --no_feat_norm "
    f"--seed 123 --device 0 --epochs 500 --patience 200 --runs 3 --display_step 100 --aggregate add --data_dir {DATA_DIR}"
)

ACTOR_BASE = (
    "python -B main.py --method pcgt --dataset film "
    "--use_graph --use_residual --backbone gcn "
    f"--seed 123 --device 0 --epochs 500 --patience 200 --runs 3 --display_step 100 --aggregate add --data_dir {DATA_DIR}"
)

print("Phase 2 attack directories ready")
print(f"  Cora:   {ATTACK_DIR}/cora/")
print(f"  Actor:  {ATTACK_DIR}/actor/")
print(f"  PubMed: {ATTACK_DIR}/pubmed/")
print(f"  Deezer: {ATTACK_DIR}/deezer/")

### CORA ATTACK (baseline: 83.80 ± 1.21 | target: 84.5+)

In [ ]:
# ──────────────────────────────────────────────────────────────
# C1: Multi-head attention (num_heads=4)
# Rationale: Citation networks benefit from multi-head; never tested >1
# ──────────────────────────────────────────────────────────────
print("\n=== C1: num_heads=4 ===")
!{CORA_BASE} --hidden_channels 64 --num_layers 4 --ours_layers 1 --num_reps 4 --num_heads 4 \
  --partition_method metis --num_partitions 7 --graph_weight 0.8 \
  --lr 0.01 --dropout 0.4 --weight_decay 5e-4 --ours_weight_decay 0.001 --ours_dropout 0.2 \
  2>&1 | tee {ATTACK_DIR}/cora/C1_heads4.txt

# ──────────────────────────────────────────────────────────────
# C2: Deeper PCGT (ours_layers=2 + residual)
# Rationale: More attention layers = deeper partition reasoning
# ──────────────────────────────────────────────────────────────
print("\n=== C2: ours_layers=2, ours_use_residual ===")
!{CORA_BASE} --hidden_channels 64 --num_layers 4 --ours_layers 2 --num_reps 4 --ours_use_residual \
  --partition_method metis --num_partitions 7 --graph_weight 0.8 \
  --lr 0.01 --dropout 0.4 --weight_decay 5e-4 --ours_weight_decay 0.001 --ours_dropout 0.2 \
  2>&1 | tee {ATTACK_DIR}/cora/C2_deep_pcgt.txt

# ──────────────────────────────────────────────────────────────
# C3: Cat aggregation (preserves both GCN + PCGT signals)
# Rationale: 'add' may lose information; 'cat' keeps both
# ──────────────────────────────────────────────────────────────
print("\n=== C3: aggregate=cat ===")
!{CORA_BASE} --hidden_channels 64 --num_layers 4 --ours_layers 1 --num_reps 4 \
  --partition_method metis --num_partitions 7 --graph_weight 0.8 --aggregate cat \
  --lr 0.01 --dropout 0.4 --weight_decay 5e-4 --ours_weight_decay 0.001 --ours_dropout 0.2 \
  2>&1 | tee {ATTACK_DIR}/cora/C3_cat.txt

In [ ]:
# ──────────────────────────────────────────────────────────────
# C4: More partitions (K=14, 2x classes)
# Rationale: 7 partitions = 7 classes, but sub-communities exist
# ──────────────────────────────────────────────────────────────
print("\n=== C4: num_partitions=14 ===")
!{CORA_BASE} --hidden_channels 64 --num_layers 4 --ours_layers 1 --num_reps 4 \
  --partition_method metis --num_partitions 14 --graph_weight 0.8 \
  --lr 0.01 --dropout 0.4 --weight_decay 5e-4 --ours_weight_decay 0.001 --ours_dropout 0.2 \
  2>&1 | tee {ATTACK_DIR}/cora/C4_K14.txt

# ──────────────────────────────────────────────────────────────
# C5: Spectral partitioning (Cora has clear spectral structure)
# Rationale: Spectral cuts may align better with class boundaries
# ──────────────────────────────────────────────────────────────
print("\n=== C5: spectral partitioning ===")
!{CORA_BASE} --hidden_channels 64 --num_layers 4 --ours_layers 1 --num_reps 4 \
  --partition_method spectral --num_partitions 7 --graph_weight 0.8 \
  --lr 0.01 --dropout 0.4 --weight_decay 5e-4 --ours_weight_decay 0.001 --ours_dropout 0.2 \
  2>&1 | tee {ATTACK_DIR}/cora/C5_spectral.txt

# ──────────────────────────────────────────────────────────────
# C6: Combined best bets (heads=4 + ours_layers=2 + K=14)
# Rationale: Stack the most promising levers
# ──────────────────────────────────────────────────────────────
print("\n=== C6: heads=4 + deep + K=14 ===")
!{CORA_BASE} --hidden_channels 64 --num_layers 4 --ours_layers 2 --num_reps 4 --num_heads 4 --ours_use_residual \
  --partition_method metis --num_partitions 14 --graph_weight 0.8 \
  --lr 0.01 --dropout 0.4 --weight_decay 5e-4 --ours_weight_decay 0.001 --ours_dropout 0.2 \
  2>&1 | tee {ATTACK_DIR}/cora/C6_combo.txt

### ACTOR ATTACK (baseline: 37.69 ± 0.98 | target: 37.9+)

In [ ]:
# ──────────────────────────────────────────────────────────────
# A1: Multi-head attention (num_heads=4)
# Rationale: Multiple attention patterns for noisy film features
# ──────────────────────────────────────────────────────────────
print("\n=== A1: num_heads=4 ===")
!{ACTOR_BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 --num_heads 4 \
  --partition_method metis --num_partitions 5 --graph_weight 0.5 \
  --lr 0.05 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 \
  2>&1 | tee {ATTACK_DIR}/actor/A1_heads4.txt

# ──────────────────────────────────────────────────────────────
# A2: More partitions (K=10) — finer-grained structure
# Rationale: 5 classes but K=5 too coarse; try K=10
# ──────────────────────────────────────────────────────────────
print("\n=== A2: num_partitions=10 ===")
!{ACTOR_BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method metis --num_partitions 10 --graph_weight 0.5 \
  --lr 0.05 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 \
  2>&1 | tee {ATTACK_DIR}/actor/A2_K10.txt

# ──────────────────────────────────────────────────────────────
# A3: Deeper PCGT (ours_layers=2 + residual)
# ──────────────────────────────────────────────────────────────
print("\n=== A3: ours_layers=2, ours_use_residual ===")
!{ACTOR_BASE} --hidden_channels 64 --num_layers 2 --ours_layers 2 --num_reps 4 --ours_use_residual \
  --partition_method metis --num_partitions 5 --graph_weight 0.5 \
  --lr 0.05 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 \
  2>&1 | tee {ATTACK_DIR}/actor/A3_deep_pcgt.txt

In [ ]:
# ──────────────────────────────────────────────────────────────
# A4: Random partitioning (worked for Deezer — another noisy dataset)
# Rationale: Film is heterophilic, METIS may not help
# ──────────────────────────────────────────────────────────────
print("\n=== A4: random partition K=10 ===")
!{ACTOR_BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method random --num_partitions 10 --graph_weight 0.5 \
  --lr 0.05 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 \
  2>&1 | tee {ATTACK_DIR}/actor/A4_random.txt

# ──────────────────────────────────────────────────────────────
# A5: Stronger regularization (d=0.6, wd=1e-3)
# Rationale: Film train acc ~55% vs test ~38%, room for regularization
# ──────────────────────────────────────────────────────────────
print("\n=== A5: dropout=0.6, wd=1e-3 ===")
!{ACTOR_BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method metis --num_partitions 5 --graph_weight 0.5 \
  --lr 0.05 --dropout 0.6 --weight_decay 1e-3 --ours_weight_decay 0.02 --ours_dropout 0.4 \
  2>&1 | tee {ATTACK_DIR}/actor/A5_reg.txt

# ──────────────────────────────────────────────────────────────
# A6: Combined best bets (heads=4 + K=10 + d=0.6)
# ──────────────────────────────────────────────────────────────
print("\n=== A6: heads=4 + K=10 + d=0.6 ===")
!{ACTOR_BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 --num_heads 4 \
  --partition_method metis --num_partitions 10 --graph_weight 0.5 \
  --lr 0.05 --dropout 0.6 --weight_decay 1e-3 --ours_weight_decay 0.02 --ours_dropout 0.4 \
  2>&1 | tee {ATTACK_DIR}/actor/A6_combo.txt

### PUBMED MARGIN WIDENING (baseline: 80.46 ± 0.64 | target: 81+)

In [ ]:
PUBMED_BASE = (
    "python -B main.py --method pcgt --dataset pubmed "
    "--use_graph --use_residual --backbone gcn --rand_split_class "
    "--label_num_per_class 20 --valid_num 500 --test_num 1000 --no_feat_norm "
    f"--seed 123 --device 0 --epochs 500 --patience 200 --runs 3 --display_step 100 --aggregate add --data_dir {DATA_DIR}"
)

# ──────────────────────────────────────────────────────────────
# P1: Multi-head attention (num_heads=4)
# ──────────────────────────────────────────────────────────────
print("\n=== P1: num_heads=4 ===")
!{PUBMED_BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 --num_heads 4 \
  --partition_method metis --num_partitions 50 --graph_weight 0.8 \
  --lr 0.01 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 \
  2>&1 | tee {ATTACK_DIR}/pubmed/P1_heads4.txt

# ──────────────────────────────────────────────────────────────
# P2: Deeper PCGT (ours_layers=2 + residual)
# ──────────────────────────────────────────────────────────────
print("\n=== P2: ours_layers=2 ===")
!{PUBMED_BASE} --hidden_channels 64 --num_layers 2 --ours_layers 2 --num_reps 4 --ours_use_residual \
  --partition_method metis --num_partitions 50 --graph_weight 0.8 \
  --lr 0.01 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 \
  2>&1 | tee {ATTACK_DIR}/pubmed/P2_deep.txt

# ──────────────────────────────────────────────────────────────
# P3: Cat aggregation + fewer partitions (K=20)
# ──────────────────────────────────────────────────────────────
print("\n=== P3: cat + K=20 ===")
!{PUBMED_BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method metis --num_partitions 20 --graph_weight 0.8 --aggregate cat \
  --lr 0.01 --dropout 0.5 --weight_decay 5e-4 --ours_weight_decay 0.01 --ours_dropout 0.3 \
  2>&1 | tee {ATTACK_DIR}/pubmed/P3_cat_K20.txt

### DEEZER MARGIN WIDENING (baseline: 67.24 ± 0.47 | target: 68+)

In [ ]:
# Build on DZ-B10 winner (random K=50, d=0.7, wd=5e-3)
DZ2_BASE = (
    "python -B main.py --method pcgt --dataset deezer-europe "
    "--use_graph --use_residual --backbone gcn --rand_split --seed 123 "
    f"--device 0 --epochs 500 --patience 200 --runs 3 --display_step 100 --aggregate add --data_dir {DATA_DIR}"
)

# ──────────────────────────────────────────────────────────────
# D1: DZ-B10 + multi-head (num_heads=4)
# ──────────────────────────────────────────────────────────────
print("\n=== D1: B10 + num_heads=4 ===")
!{DZ2_BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 --num_heads 4 \
  --partition_method random --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {ATTACK_DIR}/deezer/D1_heads4.txt

# ──────────────────────────────────────────────────────────────
# D2: DZ-B10 + deeper PCGT (ours_layers=2)
# ──────────────────────────────────────────────────────────────
print("\n=== D2: B10 + ours_layers=2 ===")
!{DZ2_BASE} --hidden_channels 64 --num_layers 2 --ours_layers 2 --num_reps 4 --ours_use_residual \
  --partition_method random --num_partitions 50 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {ATTACK_DIR}/deezer/D2_deep.txt

# ──────────────────────────────────────────────────────────────
# D3: DZ-B10 + more random partitions (K=100)
# Rationale: 28K nodes / 100 = 280 per partition, still reasonable
# ──────────────────────────────────────────────────────────────
print("\n=== D3: B10 + K=100 ===")
!{DZ2_BASE} --hidden_channels 64 --num_layers 2 --ours_layers 1 --num_reps 4 \
  --partition_method random --num_partitions 100 --graph_weight 0.5 \
  --lr 0.01 --dropout 0.7 --weight_decay 5e-3 --ours_weight_decay 0.02 --ours_dropout 0.5 \
  2>&1 | tee {ATTACK_DIR}/deezer/D3_K100.txt

### Phase 2 Results Parser

In [ ]:
import re, glob, os

baselines = {
    'cora':   ('83.80', '84.5 ± 0.8'),
    'actor':  ('37.69', '37.9 ± 1.1'),
    'pubmed': ('80.46', '80.3 ± 0.6'),
    'deezer': ('67.24', '67.1 ± 1.1'),
}

for dataset in ['cora', 'actor', 'pubmed', 'deezer']:
    our_base, sgf = baselines[dataset]
    print("=" * 70)
    print(f"  {dataset.upper()} ATTACK RESULTS  (baseline: {our_base} | SGFormer: {sgf})")
    print("=" * 70)
    print(f"{'Config':<30} {'Highest Test':<20} {'Final Test':<20}")
    print("-" * 70)

    results = []
    pattern = f"{ATTACK_DIR}/{dataset}/*.txt"
    for f in sorted(glob.glob(pattern)):
        name = os.path.basename(f).replace('.txt', '')
        with open(f) as fh:
            text = fh.read()
        m = re.search(r'Highest Test:\s+([\d.]+)\s*±\s*([\d.]+)', text)
        m2 = re.search(r'Final Test:\s+([\d.]+)\s*±\s*([\d.]+)', text)
        if m:
            ht = f"{m.group(1)} ± {m.group(2)}"
            ft = f"{m2.group(1)} ± {m2.group(2)}" if m2 else "N/A"
            results.append((name, float(m.group(1)), ht, ft))
            marker = " ★" if float(m.group(1)) > float(our_base) else ""
            print(f"{name:<30} {ht:<20} {ft:<20}{marker}")
        else:
            print(f"{name:<30} {'PARSING ERROR':<20}")

    if results:
        best = max(results, key=lambda x: x[1])
        print("-" * 70)
        print(f"BEST: {best[0]} → {best[2]}")
        improved = [r for r in results if r[1] > float(our_base)]
        print(f"Configs beating baseline: {len(improved)}/{len(results)}")
    print()

### Phase 2 Winners — 5/10-run Validation

After reviewing the 3-run probe results above, paste the winning configs here and run with full runs.

In [ ]:
# ──────────────────────────────────────────────────────────────
# PLACEHOLDER: Paste winning Cora config here with --runs 5
# ──────────────────────────────────────────────────────────────
# print("\n=== CORA BEST — 5 runs ===")
# !{CORA_BASE} <PASTE WINNING ARGS> --runs 5 2>&1 | tee {ATTACK_DIR}/cora/BEST_5run.txt

# ──────────────────────────────────────────────────────────────
# PLACEHOLDER: Paste winning Actor config here with --runs 10
# ──────────────────────────────────────────────────────────────
# print("\n=== ACTOR BEST — 10 runs ===")
# !{ACTOR_BASE} <PASTE WINNING ARGS> --runs 10 2>&1 | tee {ATTACK_DIR}/actor/BEST_10run.txt